# 🤖 Notebook 6 — Bidirectional LSTM
**Goal:** Deep learning approach using a sliding window Bidirectional LSTM. Runs both univariate and multivariate variants.

**Contents:**
1. Setup & GPU Check
2. Data Prep & Scaling
3. Sequence Generation (window=30)
4. Build Bidirectional LSTM
5. Train with Early Stopping
6. Forecast vs Actual
7. Residual Analysis
8. Metrics
9. Multivariate LSTM (bonus — same pipeline, more features)


## 1. Setup & GPU Check

In [ ]:
import os, warnings, json
os.environ["TF_CPP_MIN_LOG_LEVEL"]="3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings("ignore")
os.makedirs("plots",exist_ok=True); os.makedirs("models",exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

DATA_PATH  = "data/UPI_Master_2021_2026_Mar.csv"
TRAIN_END  = "2025-09-30"; TEST_START="2025-10-01"
WINDOW, EPOCHS, BATCH, SEED = 30, 100, 32, 42
BLUE,RED,ORANGE,GREEN,PURPLE = "#1A6FBF","#D62728","#E07B39","#2CA02C","#9467BD"
tf.random.set_seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.facecolor":"#FAFAFA","axes.facecolor":"#FAFAFA",
    "axes.grid":True,"grid.alpha":0.3,"font.family":"DejaVu Sans",
    "axes.spines.top":False,"axes.spines.right":False})

def evaluation_metrics(actual,predicted):
    a,p=np.array(actual),np.array(predicted)
    return {"MAE":round(np.mean(np.abs(a-p)),4),
            "RMSE":round(np.sqrt(np.mean((a-p)**2)),4),
            "MAPE":round(np.mean(np.abs((a-p)/a))*100,4)}


## 2. Data Prep — Univariate

In [ ]:
TARGET = "Volume (In Mn.)"   # ← change to "Value (In Cr.)" for value
MULTIVARIATE = False          # ← set True for multivariate (Section 9)

df    = pd.read_csv(DATA_PATH,parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["Month_Num"] = df["Date"].dt.month
train = df[df["Date"]<=TRAIN_END]; test=df[df["Date"]>=TEST_START]

feature_cols = [TARGET]
if MULTIVARIATE:
    feature_cols += ["Is_Weekend","Is_Festival","Is_Holiday_Adjacent",
                     "Is_Long_Weekend","Holiday_Cluster_7D","Day_Number","Month_Num"]
    print(f"Multivariate mode: {feature_cols}")
else:
    print(f"Univariate mode: {feature_cols}")

train_data = train[feature_cols].values.astype(np.float32)
test_data  = test[feature_cols].values.astype(np.float32)

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled  = scaler.transform(test_data)
print(f"Train scaled shape: {train_scaled.shape}  |  Test: {test_scaled.shape}")


## 3. Sequence Generation (sliding window = 30 days)

In [ ]:
def make_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i,0])          # predict target (col 0)
    return np.array(X), np.array(y)

combined = np.concatenate([train_scaled, test_scaled], axis=0)
X_train, y_train = make_sequences(train_scaled, WINDOW)
test_seq          = combined[len(train_scaled)-WINDOW:]
X_test,  y_test  = make_sequences(test_seq, WINDOW)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")


## 4. Build Bidirectional LSTM

In [ ]:
model = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(WINDOW, X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1),
])
model.compile(optimizer=Adam(learning_rate=1e-3), loss="huber")
model.summary()


## 5. Train with Early Stopping

In [ ]:
callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss"),
    ReduceLROnPlateau(factor=0.5, patience=7, monitor="val_loss", min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1, shuffle=False,
)

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(history.history["loss"],     color=BLUE,   lw=1.5, label="Train Loss")
ax.plot(history.history["val_loss"], color=ORANGE, lw=1.5, label="Val Loss")
ax.set_title("Training & Validation Loss (Huber)", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
plt.tight_layout(); plt.show()
print(f"Stopped at epoch {len(history.history['loss'])}")


## 6. Forecast vs Actual

In [ ]:
def inverse_target(scaled_vals):
    dummy = np.zeros((len(scaled_vals), len(feature_cols)))
    dummy[:,0] = scaled_vals
    return scaler.inverse_transform(dummy)[:,0]

y_pred_scaled = model.predict(X_test).flatten()
y_pred = inverse_target(y_pred_scaled)
y_true = inverse_target(y_test)

min_len = min(len(y_pred), len(y_true), len(test["Date"].values))
y_pred, y_true, dates = y_pred[:min_len], y_true[:min_len], test["Date"].values[:min_len]

fig, ax = plt.subplots(figsize=(16,5))
train_tail = train[TARGET].values[-60:]
ax.plot(train["Date"].values[-60:], train_tail, color=BLUE, lw=1.2, alpha=0.8, label="Train (last 60d)")
ax.plot(dates, y_true,  color=GREEN, lw=1.5, label="Actual (Test)")
ax.plot(dates, y_pred,  color=RED, lw=1.5, linestyle="--", label="LSTM Forecast")
ax.set_title(f"LSTM ({'Multivariate' if MULTIVARIATE else 'Univariate'}) — {TARGET}",
             fontsize=13, fontweight="bold")
ax.set_ylabel(TARGET); ax.legend()
plt.tight_layout()
mode_str = "multi" if MULTIVARIATE else "uni"
plt.savefig(f"plots/18_lstm_{mode_str}_{TARGET[:3].lower()}.png",dpi=150,bbox_inches="tight")
plt.show()


## 7. Residuals

In [ ]:
residuals = y_true - y_pred
fig, axes = plt.subplots(1,2,figsize=(14,4))
axes[0].plot(dates, residuals, color=ORANGE, lw=1.0)
axes[0].axhline(0,color="black",lw=1,linestyle="--")
axes[0].set_title("Residuals over Time",fontweight="bold")

import seaborn as sns
sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color=ORANGE)
axes[1].axvline(0,color="black",lw=1.5,linestyle="--")
axes[1].set_title("Residual Distribution",fontweight="bold")
plt.tight_layout(); plt.show()


## 8. Metrics & Save

In [ ]:
metrics = evaluation_metrics(y_true, y_pred)
mode_str = "multi" if MULTIVARIATE else "uni"
print(f"Test Set Metrics — LSTM ({mode_str}variate) on {TARGET}")
print(f"  MAE  : {metrics['MAE']:.4f}")
print(f"  RMSE : {metrics['RMSE']:.4f}")
print(f"  MAPE : {metrics['MAPE']:.2f}%")

with open(f"models/lstm_{mode_str}_{TARGET[:3].lower()}_metrics.json","w") as f:
    json.dump({"model":f"LSTM-BiDirectional ({mode_str}variate)",
               "target":TARGET,"window":WINDOW,"features":feature_cols,**metrics},f,indent=2)
model.save(f"models/lstm_{mode_str}_{TARGET[:3].lower()}.keras")
print("✅ Metrics & model saved.")


## 9. Multivariate LSTM
> Set `MULTIVARIATE = True` in Cell 2 and re-run all cells from Cell 2 onwards.

In [ ]:
# Quick reference — multivariate feature set:
MULTI_FEATURES = [TARGET, "Is_Weekend","Is_Festival","Is_Holiday_Adjacent",
                  "Is_Long_Weekend","Holiday_Cluster_7D","Day_Number","Month_Num"]
print("Multivariate features:", MULTI_FEATURES)
print("\nTo run: set MULTIVARIATE=True in Cell 2 and Kernel → Restart & Run All")
